# RooFit Tutorial: Introduction to Unbinned and Binned Likelihood Models

## Setup

Import ROOT and NumPy:

In [ ]:
import ROOT
import numpy as np

Silence the RooFit logging:

In [ ]:
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.FATAL)

## The basics

Mathematical concepts are represented by C++ objects:

In [ ]:
from IPython.display import Image, display
display(Image(filename="../images/roofit_classes.png"))

### Creating your first RooFit model

In [ ]:
import ROOT

Observable:

In [ ]:
x = ROOT.RooRealVar("x", "x", 0, 0, 10)

Parameters:

In [ ]:
mean = ROOT.RooRealVar("mean", "mean of gaussian", 5, 0, 10)
sigma = ROOT.RooRealVar("sigma", "width of gaussian", 1, 0.1, 10)

Gaussian PDF:

In [ ]:
gauss = ROOT.RooGaussian("gauss", "gaussian PDF", x, mean, sigma)

PDF inspection:

In [ ]:
gauss.Print("t")

### Toy dataset generation and fitting

Generate a toy dataset with 9000 entries sampled from the Gaussian PDF:

In [ ]:
data = gauss.generate({x}, 9000)

In [ ]:
data.Print()

Fit the PDF to the toy data, saving the fit result:

In [ ]:
fit_result = gauss.fitTo(data, PrintLevel=-1, Save=True)

In [ ]:
fit_result.Print()

Inspect the correlation of your model parameters:

In [ ]:
fit_result.correlationMatrix().Print()

## Plotting the data and the model

Create a `RooPlot` object on which the data and PDF is plotted:

In [ ]:
x_frame = x.frame(Title="Gaussian PDF with data")

In [ ]:
data.plotOn(x_frame)
gauss.plotOn(x_frame);

Draw the RooPlot on a `TCanvas`:

In [ ]:
c1 = ROOT.TCanvas("c1", "c1", 500, 300)
x_frame.Draw()
c1.Draw()

### Importing RooFit datasets from a ROOT file

Export the dataset to a ROOT file so we can show how to import it:

In [ ]:
data.convertToTreeStore()

output_file = ROOT.TFile("dataset.root", "RECREATE")
data.store().tree().SetName("mytree")
data.store().tree().SetTitle("My measured data")
data.store().tree().Write()
output_file.Close()

data.convertToVectorStore()

ROOT file with a TTree that stores the data you want to fit:

In [ ]:
input_file = ROOT.TFile("dataset.root", "READ") # file contains a TTree called "mytree"

Import to `RooDataSet` using the constructor that takes a TTree:

In [ ]:
dataset = ROOT.RooDataSet("dataset", "dataset", x, Import=input_file["mytree"])

Don't forget to close the file:

In [ ]:
input_file.Close()

You will see again the dataset with the observable `x`:

In [ ]:
dataset.Print()

### Exporting your RooFit datasets

You can export a RooDataSet to NumPy or Pandas:

In [ ]:
df = data.to_pandas()

In [ ]:
df

## Composite PDFs

Composite PDF: model with mutiple components, like signal and background.

This time, we create a toy data set with RDataFrame, like you would in a real analysis:

In [ ]:
df = ROOT.RDataFrame(45000).Define(
    "x",
    "rdfentry_ < 9000 ? gRandom->Gaus(5.0, 1.0) : gRandom->Exp(1.0 / 0.18)"
)
print("Total entries:", df.Count().GetValue())

Create the RooDataSet via a RDataFrame helper action (as explained in [this tutorial](https://root.cern/doc/master/rf408__RDataFrameToRooFit_8py.html)):

In [ ]:
data_x = df.Book(
    ROOT.std.move(ROOT.RooDataSetHelper("my_data", "", ROOT.RooArgSet(x))), ("x",)
).GetValue()
data_x.Print() # events outside RooRealVar range will be dropped

Visualize the dataset:

In [ ]:
x_frame = x.frame(Title="Plotting Gaussian plus exp. background")
data_x.plotOn(x_frame)

c2 = ROOT.TCanvas()
x_frame.Draw()
c2.Draw()

### Creating the composite fit model

Create exponential PDF with parameter "tau":

In [ ]:
tau = ROOT.RooRealVar("tau", "tau", -0.2, -10.0, -0.01)
expo = ROOT.RooExponential("expo", "expo", x, tau)

Define parameters for the number of signal and background events:

In [ ]:
n_sig = ROOT.RooRealVar("n_sig", "n_sig", 10000, 1000, 100000)
n_bkg = ROOT.RooRealVar("n_bkg", "n_bkg", 50000, 5000, 500000)

Composite model that automatically includes a Poisson term for the total number of events:

In [ ]:
model = ROOT.RooAddPdf("model", "model", [gauss, expo], [n_sig, n_bkg])

Do the fit:

In [ ]:
fit_result = model.fitTo(data_x, PrintLevel=-1, Save=True)
fit_result.Print()

## Creating a nice plot

Create RooPlot and draw data, PDF, and components:

In [ ]:
x_frame = x.frame(Title="Gaussian plus exp. background")

data_x.plotOn(x_frame, Name="data")

model.plotOn(x_frame, Components=gauss, LineColor="r", LineStyle="--", Name="gauss")
model.plotOn(x_frame, Components=expo, LineColor="k", LineStyle="--", Name="expo")
model.plotOn(x_frame, Name="model");

Add a legend:

In [ ]:
legend = ROOT.TLegend(0.7, 0.55, 0.92, 0.87)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.AddEntry(x_frame.findObject("data"), "data", "P")

for name in ["model", "gauss", "expo"]:
    legend.AddEntry(x_frame.findObject(name), name, "L")

Create a second frame with the residuals:

In [ ]:
resid_hist = x_frame.residHist()

resid_frame = x.frame(Title=";x;residuals")
resid_frame.addPlotable(resid_hist, "P")

Create a canvas that is divided into two drawing pads:

In [ ]:
c3 = ROOT.TCanvas("c3", "c3", 600, 600)
c3.Divide(1, 2)

First pad is for the main plot and the legend:

In [ ]:
pad_1 = c3.cd(1)
x_frame.Draw()
legend.Draw()
pad_1.SetPad(0.0, 0.2, 1, 1)

Second pad is for the residuals:

In [ ]:
pad_2 = c3.cd(2)
pad_2.SetPad(0., 0.0, 1, 0.25)
resid_frame.Draw()
resid_frame.GetXaxis().SetLabelSize(0.12)
resid_frame.GetYaxis().SetLabelSize(0.12)
resid_frame.GetYaxis().SetTitleSize(0.12)
resid_frame.GetYaxis().SetTitleOffset(0.25)

Draw the canvas:

In [ ]:
c3.Draw()

## Template fits with convolutions

A **template** PDF is based on *histogram shape*, and not expressed by an analytical function.

Imagine you have a histogram giving the expected signal shape:

In [ ]:
template_hist = ROOT.TH1D("h1", "h1", 100, 0, 10)
f1 = ROOT.TF1("f1", "std::exp(-std::abs((x-5)))", 0, 10)
template_hist.FillRandom("f1", 100000)

In [ ]:
c4 = ROOT.TCanvas("c4", "c4", 600, 400)
template_hist.Draw()
c4.Draw()

Getting the template PDF into RooFit:

1. Create a corresponding observable:

In [ ]:
y = ROOT.RooRealVar("y", "y", 0, 0, 10)

2. Convert the `TH1` into a `RooDataHist`:

In [ ]:
roo_template_hist = ROOT.RooDataHist("roo_template_hist", "roo_template_hist", y, template_hist)

3. Create a `RooHistPdf` based of the RooFit histogram:

In [ ]:
sig_raw_y = ROOT.RooHistPdf("sig__raw_y", "sig_raw_y", y, roo_template_hist)

### Creating a full composite model

Construct a `RooGaussian` to model detector resolution effects:

In [ ]:
resolution = ROOT.RooRealVar("resolution", "resolution", 0.2, 0.1, 1.0)
sig_smearing_y = ROOT.RooGaussian("sig_smearing_y", "sig_smearing_y", y, ROOT.RooFit.RooConst(0.0), resolution)

The signal PDF is a convolution of the template and the resolution function:

In [ ]:
sig_y = ROOT.RooFFTConvPdf("sig_y", "sig_y", y, sig_raw_y, sig_smearing_y)

For the background, we use a Chebychev polynomial:

In [ ]:
bkg_y = ROOT.RooChebychev("bkg_y", "bkg_y", y, [-0.5, 0.1])

Finally, create **RooAddPdf** for the composite model:

In [ ]:
model_y = ROOT.RooAddPdf("model_y", "model_x", [sig_y, bkg_y], [n_sig, n_bkg])

Creating toy dataset and fitting

In [ ]:
data_y = model_y.generate(y)

In [ ]:
fit_result = model_y.fitTo(data_y, PrintLevel=-1, Save=True)
fit_result.Print()

Plotting the model and the toy dataset

In [ ]:
y_frame = y.frame(Title="Model for y")

data_y.plotOn(y_frame)
model_y.plotOn(y_frame)

c5 = ROOT.TCanvas()
y_frame.Draw()
c5.Draw()

### Overview of other PDF types

RooFit provides a collection of standard PDF classes, e.g.:

In [ ]:
from IPython.display import Image, display
display(Image(filename="../images/roofit_pdfs.png"))

Easy to **extend the library**: each pdf is a separate C++ class

## Multivariate fit

We have now modeled two observables:
* `x` with Gaussian signal and exponential background
* `y` with smeared template signal and Chebychev background

Create 2D model $p(x,y) = p(x) p(y)$ for signal and background

In [ ]:
model_sig_xy = ROOT.RooProdPdf("model_sig_xy", "model_sig_xy", [gauss, sig_y])
model_bkg_xy = ROOT.RooProdPdf("model_bkg_xy", "model_bkg_xy", [expo, bkg_y])

Yet again, a RooAddPdf for the final model

In [ ]:
model_xy = ROOT.RooAddPdf("model_xy", "model_xy", [model_sig_xy, model_bkg_xy], [n_sig, n_bkg])

Generating a 2D toy dataset

In [ ]:
data_xy = model_xy.generate({x, y}, 10000)

Visualize the 2D data in a LEGO plot

In [ ]:
histo_xy = data_xy.createHistogram("histo_xy", x, Binning=25, YVar=dict(var=data_xy.get()["y"], Binning=15))
histo_xy.SetTitle("")

c6 = ROOT.TCanvas()
histo_xy.Draw("LEGO2")
c6.Draw()

Fitting the 2D model

By now you know how it works:

In [ ]:
fit_result_xy = model_xy.fitTo(data_xy, PrintLevel=-1, Save=True)

Fit result has all parameters we saw before:

In [ ]:
fit_result_xy.Print()

### Final visualization of 2D model and data

In [ ]:
x_frame = x.frame(Title="Model for x")
y_frame = y.frame(Title="Model for y")

data_xy.plotOn(x_frame)
model_xy.plotOn(x_frame)

data_xy.plotOn(y_frame)
model_xy.plotOn(y_frame)

c7 = ROOT.TCanvas("c7", "c7", 800, 400)
c7.Divide(2)
c7.cd(1)
x_frame.Draw()
c7.cd(2)
y_frame.Draw()
c7.Draw()

### Likelihood scans

It is very useful to plot and inspect the NLL and also the profiled NLL. For this, you can use `createNLL` to get a RooFit object that represents a likelihood directly. More examples can be found in the [rf605_profilell tutorial](https://root.cern/doc/master/rf605__profilell_8py.html).

In [ ]:
# Create likelihood function
nll = model_xy.createNLL(data_xy, EvalBackend="cpu")
# The new "cpu" evaluation backend is a performance optimization, it can also be used in `fitTo`

# Minimize likelihood such that all other parameters (nuisance parameters) are at the best fit value.
minimizer = ROOT.RooMinimizer(nll)
minimizer.setPrintLevel(-1)
minimizer.minimize("Minuit", "")

# Make RooPlot for our parameter of interest, let's say n_sig
window = 5 * n_sig.getError()
n_sig_frame = n_sig.frame(Bins=10, Range=(n_sig.getVal() - window, n_sig.getVal() + window))

# Plot likelihood scan in parameter n_sig
nll.plotOn(n_sig_frame, ShiftToZero=True)

# Plot the profile likelihood in n_sig.
# Now, the nuisance parameter are optimized for each scanned value of n_sig.
pll_n_sig = nll.createProfile([n_sig])
pll_n_sig.plotOn(n_sig_frame, LineColor="kRed")

# Set y axis limits
n_sig_frame.SetMinimum(0)
n_sig_frame.SetMaximum(5)

c8 = ROOT.TCanvas()
n_sig_frame.Draw()
c8.Draw()

## Model inspection

You already know the `Print("t")` function:

In [ ]:
model.Print("t")

## The RooWorkspace

The RooFit objects can be managed by a `RooWorkspace`:

In [ ]:
ws = ROOT.RooWorkspace("myworkspace")

You can for example import an existing model:

In [ ]:
ws.Import(model_xy);

You can `Print` the workspace for inspecting its content:

In [ ]:
ws.Print()

Access any object in the RooWorkspace:

In [ ]:
ws["model_xy"].Print()

Save the workspace to a ROOT file to reuse the model later

In [ ]:
ws.writeToFile("myworkspace.root");

## Binned models and HistFactory

So far, we did **unbinned** fits with models that have an analytical shape (possibly convoluted with a template).

In LHC analyses, one often performs **binned** likelihood fits instead: the model predicts the expected number of events in each bin of a histogram, and the prediction for each sample is usually taken from a **Monte Carlo template histogram**.

Systematic uncertainties are implemented with **nuisance parameters** that continuously *interpolate* between the nominal template and systematically varied templates, changing the normalization or even the shape of the prediction. Each nuisance parameter is accompanied by a constraint term in the likelihood.

Even for a single channel with a few samples, spelling out such a likelihood with individual RooFit objects is tedious and error-prone. Therefore, binned models are usually created with **higher-level frameworks on top of RooFit**. The framework that ships with ROOT is [HistFactory](https://root.cern/doc/master/group__HistFactory.html).

We will now build the model from the [hf001_example.py tutorial](https://root.cern/doc/master/hf001__example_8py.html): one channel with a signal sample and two background samples.

First, we create the template and data histograms and save them to a ROOT file, just like the histogram-making step of a real analysis would:

In [ ]:
input_file_name = "hf_input.root"

h_sig = ROOT.TH1D("signal", "signal template", 2, 1, 2)
h_sig.SetBinContent(1, 20)
h_sig.SetBinContent(2, 10)

h_bkg1 = ROOT.TH1D("background1", "background 1 template", 2, 1, 2)
h_bkg1.SetBinContent(1, 100)

h_bkg2 = ROOT.TH1D("background2", "background 2 template", 2, 1, 2)
h_bkg2.SetBinContent(2, 100)

# Relative statistical uncertainties of the background 1 template:
h_bkg1_statuncert = ROOT.TH1D("background1_statUncert", "background 1 rel. uncert.", 2, 1, 2)
h_bkg1_statuncert.SetBinContent(1, 0.05)
h_bkg1_statuncert.SetBinContent(2, 0.05)

# The observed data:
h_data = ROOT.TH1D("data", "data", 2, 1, 2)
h_data.SetBinContent(1, 122)
h_data.SetBinContent(2, 112)

with ROOT.TFile.Open(input_file_name, "RECREATE") as hf_input_file:
    for hist in [h_sig, h_bkg1, h_bkg2, h_bkg1_statuncert, h_data]:
        hf_input_file.WriteObject(hist, hist.GetName())

A HistFactory model is declared with a `Measurement` object. We define the parameter of interest — the signal strength `SigXsecOverSM` — and the integrated luminosity with its uncertainty. Like in the original tutorial, the luminosity and the nuisance parameter for the signal systematic are kept constant in the fit:

In [ ]:
meas = ROOT.RooStats.HistFactory.Measurement("meas", "meas")

meas.SetPOI("SigXsecOverSM")
meas.SetLumi(1.0)
meas.SetLumiRelErr(0.10)

meas.AddConstantParam("Lumi")
meas.AddConstantParam("alpha_syst1")

A measurement contains one or more **channels**, i.e. disjoint regions of the data like signal or control regions. Each channel gets its observed data and a configuration for the statistical uncertainties of the templates (here: ignore them below 5 % relative uncertainty, and use Poisson constraint terms):

In [ ]:
chan = ROOT.RooStats.HistFactory.Channel("channel1")
chan.SetData("data", input_file_name)
chan.SetStatErrorConfig(0.05, "Poisson")

Each channel contains **samples**, whose expected distributions are given by the template histograms. The signal sample gets a free normalization factor (our parameter of interest) and a ±5 % normalization uncertainty called `syst1`:

In [ ]:
signal = ROOT.RooStats.HistFactory.Sample("signal", "signal", input_file_name)
signal.AddOverallSys("syst1", 0.95, 1.05)
signal.AddNormFactor("SigXsecOverSM", 1, 0, 3)
chan.AddSample(signal)

The background samples get normalization uncertainties too. In addition, we activate the **statistical uncertainty of the templates** themselves: for `background1` it is read from the dedicated histogram of relative uncertainties we created above, and for `background2` it is taken from the bin errors of the template. HistFactory turns this into one nuisance parameter per bin, shared by all samples in the channel and constrained by the configured Poisson terms (also known as the *Barlow–Beeston method*).

If you also want a systematic variation to change the *shape* of a template, you would use `AddHistoSys()`, which takes a down- and up-varied histogram to interpolate between — see the [HistFactory documentation](https://root.cern/doc/master/group__HistFactory.html).

In [ ]:
background1 = ROOT.RooStats.HistFactory.Sample("background1", "background1", input_file_name)
background1.ActivateStatError("background1_statUncert", input_file_name)
background1.AddOverallSys("syst2", 0.95, 1.05)
chan.AddSample(background1)

background2 = ROOT.RooStats.HistFactory.Sample("background2", "background2", input_file_name)
background2.ActivateStatError()
background2.AddOverallSys("syst3", 0.95, 1.05)
chan.AddSample(background2)

Add the channel to the measurement, collect the histograms from the input file, and inspect the complete measurement:

In [ ]:
meas.AddChannel(chan)
meas.CollectHistograms()
meas.PrintTree()

So far, everything was purely *declarative* — no RooFit objects were involved yet. Now we let HistFactory build the actual model from the measurement specification:

In [ ]:
ws_hf = ROOT.RooStats.HistFactory.HistoToWorkspaceFactoryFast.MakeCombinedModel(meas)

The result is a `RooWorkspace`, which you already know. Printing it shows how much work HistFactory did for us: the templates became `RooHistFunc` objects, the systematics became `FlexibleInterpVar` interpolation objects with `RooGaussian` constraint terms, the template statistics became one `gamma_stat_*` parameter per bin with Poisson constraints, and everything is combined into a `RooSimultaneous` PDF:

In [ ]:
ws_hf.Print()

The workspace also contains a `ModelConfig` object that documents the model for the statistics tools in RooStats: which PDF to use, which parameters are the parameters of interest, which are the observables, and which are the *global observables* of the constraint terms:

In [ ]:
model_config = ws_hf["ModelConfig"]
pdf_hf = model_config.GetPdf()

Since this is an ordinary RooFit PDF, we can fit it to the observed data like any other model. For models with constraint terms, it is recommended to explicitly pass the global observables:

In [ ]:
fit_result_hf = pdf_hf.fitTo(
    ws_hf["obsData"],
    GlobalObservables=model_config.GetGlobalObservables(),
    PrintLevel=-1,
    Save=True,
)
fit_result_hf.Print()

All the tools from before work here as well, for example the profile likelihood scan of the signal strength, where all the nuisance parameters that HistFactory created are profiled:

In [ ]:
poi = model_config.GetParametersOfInterest().first()

nll_hf = pdf_hf.createNLL(ws_hf["obsData"], EvalBackend="cpu")
pll_hf = nll_hf.createProfile([poi])

poi_frame = poi.frame(Title="Profile likelihood scan of the signal strength")
nll_hf.plotOn(poi_frame, ShiftToZero=True, LineColor="kRed", LineStyle="--")
pll_hf.plotOn(poi_frame)
poi_frame.GetYaxis().SetTitle("-log likelihood")
poi_frame.SetMinimum(0)
poi_frame.SetMaximum(3)

c9 = ROOT.TCanvas("c9", "c9", 500, 300)
poi_frame.Draw()
c9.Draw()

### Higher-level frameworks in the wild

The takeaway: binned likelihood fits are typically built from Monte Carlo templates that are interpolated according to nuisance parameters, and such models are complicated enough that nobody writes them out by hand. Higher-level frameworks generate them from a declarative specification.

We have seen a simple example with HistFactory, but the LHC experiments often use their own high-level frameworks for this:

- [CMS Combine](https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/): the CMS statistics framework, built on top of RooFit and RooStats
- [TRExFitter](https://trexfitter-docs.web.cern.ch/trexfitter/): widely used in ATLAS, generates HistFactory workspaces from configuration files
- [pyhf](https://pyhf.readthedocs.io/): a pure-Python implementation of the HistFactory model specification, independent of ROOT

They differ in configuration language and features, but underneath they all describe the same kind of binned likelihood model that you now know how to build, inspect, and fit yourself.

## Exercises

1. Further improve the plot with the pull distribution by visualizing also the post-fit uncertainty of the model. Figure out how to do this by reading the documentation of [RooAbsPdf::plotOn()](https://root.cern.ch/doc/master/classRooAbsPdf.html#aa0f2f98d89525302a06a1b7f1b0c2aa6).

2. Look at the [rf203_ranges.py RooFit tutorial](https://root.cern/doc/master/rf203__ranges_8py.html) to learn how to restrict the fit to a subrange. Redo the convoluted template fit to the $y$ variable, but restricted to the range from 3 to 7.

   Why does the uncertainty of the `resolution` parameter increase, even though we are not excluding that much signal and `resolution` doesn't affect the background?

3. Interpret the likelihood plot over `n_sig`. Why is the profile NLL always below the other plotted NLL?

4. In a fresh notebook, open the `RooWorkspace` we wrote to disk and create new toy data according to the 2D model. Re-fit the model to the new toy dataset.

5. Which parameters are strongly (anti)correlated in the final 2D fit? Can you explain why?

6. For the multidimensional model, why did we not just create a single `RooProdPdf` that multiples the model for $x$ and the model for $y$?

7. In the HistFactory example, the nuisance parameter `alpha_syst1` of the signal normalization uncertainty was set constant. Remove the corresponding `AddConstantParam()` call and rebuild the model. How do the fitted value and the uncertainty of `SigXsecOverSM` change? What happens if you increase `syst1` to a ±20 % uncertainty?